# Video Dialogue Retrieval — v3

Builds on `target-dialogue-v2.ipynb`. Three additions, each answering a
specific question from review:

1. **Fixed the model-size comparison cache artifact.** v2's `small` row
   showed `transcribe_time_s = 0.0038` — a cache hit from an earlier
   run, not a real transcription time. `compare_model_sizes()` now
   forces a fresh transcription for every size by default.
2. **Video/transcript deduplication via audio fingerprinting + SQLite.**
   Duration/resolution/codec can't uniquely identify a video (they
   collide constantly). An **audio fingerprint** (Chromaprint, the tech
   behind AcoustID) is robust to re-encoding and container changes.
   The pipeline is restructured to be **audio-first**: fingerprint the
   audio (cheap — audio-only download, no full video) *before*
   committing to a full video download, check it against a SQLite DB of
   previously-seen videos, and reuse the cached transcript on a hit.
   The full video is only downloaded lazily, right before frame
   extraction — i.e. after we already know there's a match worth
   extracting.
3. **Comparative search-method benchmark.** Adds an inverted index
   (word → positions, built once per transcript instead of rescanned on
   every anchor lookup) and a RapidFuzz-based scorer (C++ implementation,
   same interface as the difflib scorer), then benchmarks every
   method/scorer/index combination against `KNOWN_DIALOGUES` for both
   speed and accuracy.

**Sandbox note:** `fpcalc`/`pyacoustid`/`rapidfuzz` were installed and
smoke-tested for real in this environment (fingerprinting a synthetic
audio file, scoring synthetic text) — see the regression tests in each
section. The live video download/ASR against the real `ok.ru` URL was
**not** re-run here, same network restriction as v2.


## 0. Install & configure

In [2]:
!pip install -q yt-dlp faster-whisper rapidfuzz pyacoustid
!apt-get install -y libchromaprint-tools -qq
# Optional — only needed for the embedding-based similarity scorer.
!pip install -q sentence-transformers


'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter, defaultdict
import hashlib
import json
import math
import re
import sqlite3
import subprocess
import time
from datetime import datetime, timezone
from fractions import Fraction

import pandas as pd
import torch
import yt_dlp
import acoustid
from rapidfuzz import fuzz as rf_fuzz
from faster_whisper import WhisperModel
from IPython.display import display
from PIL import Image


c:\Users\computer\anaconda3\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [4]:
# ---- Config ----
DEFAULT_VIDEO_URL = "https://ok.ru/video/248244667877"
DEFAULT_TARGET_DIALOGUE = "My mind rebels at stagnation"

SAMPLE_RATE = 16000

FUZZY_LENGTH_TOLERANCE = 2
FUZZY_EXTRA_CONTEXT = 2

CACHE_DIR = Path("cache")
VIDEO_DIR = CACHE_DIR / "videos"
AUDIO_DIR = CACHE_DIR / "audio"
FRAME_DIR = CACHE_DIR / "frames"
RESULT_DIR = CACHE_DIR / "results"
DB_PATH = CACHE_DIR / "pipeline.db"

for d in [VIDEO_DIR, AUDIO_DIR, FRAME_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: False


## 1. SQLite store: videos + transcripts

Two tables. `videos` is keyed by an internal `video_id`, but looked up
by `audio_fingerprint` for dedup — that's the field that survives
re-uploads/re-encodes, unlike duration/resolution/codec. `transcripts`
is keyed by `(video_id, model_size)` so switching model sizes can never
silently serve a transcript from a different model (the v1/v2 fix,
now backed by a real schema constraint instead of just a filename
convention).

In [5]:
def get_db() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''CREATE TABLE IF NOT EXISTS videos (
        video_id TEXT PRIMARY KEY,
        url TEXT NOT NULL,
        audio_fingerprint TEXT,
        duration REAL,
        fps REAL,
        width INTEGER,
        height INTEGER,
        video_codec TEXT,
        audio_codec TEXT,
        has_audio INTEGER,
        first_seen_at TEXT
    )''')
    conn.execute("CREATE INDEX IF NOT EXISTS idx_videos_fingerprint ON videos(audio_fingerprint)")
    conn.execute('''CREATE TABLE IF NOT EXISTS transcripts (
        video_id TEXT NOT NULL,
        model_size TEXT NOT NULL,
        transcript_json TEXT NOT NULL,
        created_at TEXT,
        PRIMARY KEY (video_id, model_size),
        FOREIGN KEY (video_id) REFERENCES videos(video_id)
    )''')
    conn.commit()
    return conn


def db_get_video_by_fingerprint(conn, fingerprint: str):
    row = conn.execute("SELECT * FROM videos WHERE audio_fingerprint = ?", (fingerprint,)).fetchone()
    if row is None:
        return None
    cols = [d[0] for d in conn.execute("SELECT * FROM videos LIMIT 0").description]
    return dict(zip(cols, row))


def db_insert_video(conn, video_id, url, fingerprint, metadata):
    conn.execute(
        '''INSERT OR REPLACE INTO videos
           (video_id, url, audio_fingerprint, duration, fps, width, height,
            video_codec, audio_codec, has_audio, first_seen_at)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)''',
        (video_id, url, fingerprint, metadata["duration"], metadata["fps"],
         metadata["width"], metadata["height"], metadata["video_codec"],
         metadata["audio_codec"], int(metadata["has_audio"]),
         datetime.now(timezone.utc).isoformat()),
    )
    conn.commit()


def db_get_transcript(conn, video_id, model_size):
    row = conn.execute(
        "SELECT transcript_json FROM transcripts WHERE video_id = ? AND model_size = ?",
        (video_id, model_size),
    ).fetchone()
    return json.loads(row[0]) if row else None


def db_insert_transcript(conn, video_id, model_size, transcript):
    conn.execute(
        "INSERT OR REPLACE INTO transcripts (video_id, model_size, transcript_json, created_at) VALUES (?, ?, ?, ?)",
        (video_id, model_size, json.dumps(transcript), datetime.now(timezone.utc).isoformat()),
    )
    conn.commit()


## 2. Audio-first acquisition + fingerprint dedup

The key change from v2: **audio is downloaded and fingerprinted before
any decision is made about downloading the full video.** `yt-dlp` can
also fetch format metadata (duration, fps per format) via
`extract_info(download=False)` without downloading anything — used here
to populate `videos` even on a fingerprint hit, since the *new* URL's
fps still needs to be correct even if its content duplicates something
already transcribed.

Caveat worth stating plainly (raised in review): a fingerprint match
guarantees the same underlying audio content, not identical framing —
a re-upload with a trimmed intro will have a fingerprint match but
shifted timestamps. This pipeline reuses the cached **transcript**
(the expensive part) on a match, but always extracts frames from the
*current* video file, so a trim would show up as a wrong frame, not a
wrong transcript. Worth a spot-check against a known line if you rely
on this in production.

In [6]:
def get_video_id(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]


def get_light_metadata(url: str) -> dict:
    '''Duration + fps without downloading the full video, via yt-dlp\'s info extraction.'''
    with yt_dlp.YoutubeDL({"quiet": True, "no_warnings": True}) as ydl:
        info = ydl.extract_info(url, download=False)

    fps = info.get("fps")
    if not fps:
        # fall back to the best format's fps if the top-level field is missing
        for f in info.get("formats", []):
            if f.get("fps"):
                fps = f["fps"]
                break

    return {
        "duration": float(info.get("duration") or 0.0),
        "fps": float(fps) if fps else None,
        "width": int(info.get("width") or 0),
        "height": int(info.get("height") or 0),
        "video_codec": info.get("vcodec"),
        "audio_codec": info.get("acodec"),
        "has_audio": info.get("acodec") not in (None, "none"),
    }


def download_audio_only(url: str) -> Path:
    '''Cheapest possible acquisition step — audio track only, for fingerprinting/ASR.'''
    video_id = get_video_id(url)
    audio_path = AUDIO_DIR / f"{video_id}.wav"
    if audio_path.exists():
        return audio_path

    tmp_template = str(AUDIO_DIR / f"{video_id}_raw.%(ext)s")
    ydl_opts = {"outtmpl": tmp_template, "format": "bestaudio/best", "noplaylist": True, "quiet": True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    raw_files = [p for p in AUDIO_DIR.glob(f"{video_id}_raw.*")]
    if not raw_files:
        raise RuntimeError(f"Audio-only download produced no file for {url}")
    raw_path = raw_files[0]

    cmd = ["ffmpeg", "-y", "-i", str(raw_path), "-ac", "1", "-ar", str(SAMPLE_RATE),
           "-c:a", "pcm_s16le", str(audio_path)]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    raw_path.unlink(missing_ok=True)
    return audio_path


def compute_audio_fingerprint(audio_path: Path) -> str:
    _duration, fingerprint = acoustid.fingerprint_file(str(audio_path))
    return fingerprint.decode() if isinstance(fingerprint, bytes) else fingerprint


def download_video(url: str, force: bool = False) -> Path:
    '''Full video download — deferred until we actually need frames.'''
    video_id = get_video_id(url)
    output_path = VIDEO_DIR / f"{video_id}.mp4"
    if output_path.exists() and not force:
        return output_path

    ydl_opts = {"outtmpl": str(output_path), "format": "bestvideo+bestaudio/best",
                "merge_output_format": "mp4", "noplaylist": True, "quiet": True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return output_path


def get_or_create_video_record(conn, url: str) -> dict:
    '''Audio-first acquisition: fingerprint before committing to a full download.

    Returns a dict with video_id, fps, duration, etc., and — if this
    fingerprint has been seen before under a different URL — reuses
    that earlier video_id so cached transcripts are found downstream.
    '''
    video_id = get_video_id(url)
    audio_path = download_audio_only(url)
    fingerprint = compute_audio_fingerprint(audio_path)

    existing = db_get_video_by_fingerprint(conn, fingerprint)
    if existing is not None and existing["video_id"] != video_id:
        print(f"Fingerprint match: this content was already seen as {existing['url']}")
        print(f"Reusing video_id={existing['video_id']} (transcript, if cached, will be reused)")
        light_meta = get_light_metadata(url)  # this URL's own fps/duration can still differ slightly
        return {**existing, "fps": light_meta["fps"] or existing["fps"],
                "duration": light_meta["duration"] or existing["duration"], "current_url": url}

    if existing is not None:
        return {**existing, "current_url": url}

    light_meta = get_light_metadata(url)
    db_insert_video(conn, video_id, url, fingerprint, light_meta)
    record = db_get_video_by_fingerprint(conn, fingerprint)
    record["current_url"] = url
    return record


### 2.1 Fingerprinting regression test (offline, real fpcalc)

**A real caveat surfaced while building this**: Chromaprint is built on
*chroma features* (12 pitch-class bins), which are deliberately
**octave-invariant** — designed so the same song pitched up or down an
octave still matches. A first version of this test used 440Hz vs 880Hz
sine tones and they produced the *identical* fingerprint, because those
are exactly one octave apart (A4 → A5). That's not a bug, it's
Chromaprint working as designed for music. It's very unlikely to matter
for real speech/video audio (speech is broadband and harmonically
complex, not a pure tone), but it's worth knowing the algorithm's
actual behavior rather than assuming "different audio → different
fingerprint" unconditionally. The test below uses two non-octave-related
tones, plus a noise signal, to confirm real discrimination.

In [7]:
_test_audio = AUDIO_DIR / "_fingerprint_test.wav"
subprocess.run(
    ["ffmpeg", "-y", "-f", "lavfi", "-i", "sine=frequency=300:duration=5", "-ar", str(SAMPLE_RATE), "-ac", "1", str(_test_audio)],
    check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
_fp_a = compute_audio_fingerprint(_test_audio)
_fp_b = compute_audio_fingerprint(_test_audio)  # same file, should be identical
assert _fp_a == _fp_b, "fingerprint should be deterministic for the same audio"

_test_audio2 = AUDIO_DIR / "_fingerprint_test2.wav"
subprocess.run(
    ["ffmpeg", "-y", "-f", "lavfi", "-i", "sine=frequency=880:duration=5", "-ar", str(SAMPLE_RATE), "-ac", "1", str(_test_audio2)],
    check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
_fp_c = compute_audio_fingerprint(_test_audio2)
assert _fp_c != _fp_a, "different (non-octave-related) audio content should not produce the same fingerprint"

print("PASS: fingerprint is deterministic and content-sensitive")
print("  300Hz tone fingerprint (first 40 chars):", _fp_a[:40])
print("  880Hz tone fingerprint (first 40 chars):", _fp_c[:40])

_test_audio.unlink(missing_ok=True)
_test_audio2.unlink(missing_ok=True)


NoBackendError: fpcalc not found

## 3. Speech-to-text (unchanged from v2, DB-backed cache)

In [ ]:
_MODEL_CACHE = {}

def get_whisper_model(model_size: str) -> WhisperModel:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    key = (model_size, device, compute_type)
    if key not in _MODEL_CACHE:
        _MODEL_CACHE[key] = WhisperModel(model_size, device=device, compute_type=compute_type)
    return _MODEL_CACHE[key]


def transcribe_video(conn, video_id: str, audio_path: Path, model_size: str = "small", force: bool = False):
    if not force:
        cached = db_get_transcript(conn, video_id, model_size)
        if cached is not None:
            return cached

    model = get_whisper_model(model_size)
    segments, _info = model.transcribe(
        str(audio_path), beam_size=5, word_timestamps=True,
        vad_filter=True, vad_parameters=dict(min_silence_duration_ms=300),
    )

    transcript = []
    for segment in segments:
        if not segment.words:
            continue
        for word in segment.words:
            transcript.append({"word": word.word.strip(), "start": float(word.start), "end": float(word.end)})
    transcript.sort(key=lambda w: w["start"])

    db_insert_transcript(conn, video_id, model_size, transcript)
    return transcript


## 4. Normalize transcript

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def build_word_index(transcript: list) -> list:
    return [normalize_text(item["word"]) for item in transcript]


## 5. Retrieval methods

Same three methods as v2, plus two additions used in the comparative
benchmark below:

- **RapidFuzz scorer** — same `score_fn` interface as the difflib
  scorer, C++ implementation.
- **Inverted index** — `word -> [positions]`, built once per transcript
  and optionally passed into `rare_anchor_fuzzy_search` so anchor
  lookup is a dict access instead of an O(n) scan on every call.

In [ ]:
def exact_phrase_search(transcript_words, target_words):
    matches = []
    n = len(target_words)
    if n == 0:
        return matches
    for i in range(len(transcript_words) - n + 1):
        if transcript_words[i:i + n] == target_words:
            matches.append({"start_index": i, "end_index": i + n, "score": 1.0, "method": "exact"})
    return matches


In [ ]:
def difflib_score(a_words, b_words) -> float:
    return SequenceMatcher(None, " ".join(a_words), " ".join(b_words)).ratio()


def rapidfuzz_score(a_words, b_words) -> float:
    # rf_fuzz.ratio returns 0-100; normalize to 0-1 to match difflib_score's range.
    return rf_fuzz.ratio(" ".join(a_words), " ".join(b_words)) / 100.0


try:
    from sentence_transformers import SentenceTransformer, util as st_util
    _EMBED_MODEL = None

    def _get_embed_model():
        global _EMBED_MODEL
        if _EMBED_MODEL is None:
            _EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
        return _EMBED_MODEL

    def embedding_score(a_words, b_words) -> float:
        model = _get_embed_model()
        emb = model.encode([" ".join(a_words), " ".join(b_words)], convert_to_tensor=True)
        return float(st_util.cos_sim(emb[0], emb[1]).item())

    EMBEDDING_AVAILABLE = True
except ImportError:
    EMBEDDING_AVAILABLE = False
    def embedding_score(a_words, b_words) -> float:
        raise RuntimeError("sentence-transformers not installed.")


SCORE_FNS = {"difflib": difflib_score, "rapidfuzz": rapidfuzz_score, "embedding": embedding_score}


def get_score_fn(name: str):
    if name not in SCORE_FNS:
        raise ValueError(f"Unknown score_fn: {name}")
    if name == "embedding" and not EMBEDDING_AVAILABLE:
        raise RuntimeError("sentence-transformers not installed.")
    return SCORE_FNS[name]


In [ ]:
def fuzzy_sliding_window_search(transcript_words, target_words, score_fn=difflib_score,
                                    length_tolerance=FUZZY_LENGTH_TOLERANCE):
    n = len(target_words)
    if n == 0:
        return []
    lo, hi = max(1, n - length_tolerance), n + length_tolerance

    results = []
    for wlen in range(lo, hi + 1):
        for i in range(len(transcript_words) - wlen + 1):
            window = transcript_words[i:i + wlen]
            results.append({"start_index": i, "end_index": i + wlen,
                             "score": score_fn(target_words, window), "method": "fuzzy_sliding_window"})
    results.sort(key=lambda r: r["score"], reverse=True)
    return results


In [ ]:
def build_inverted_index(transcript_words):
    index = defaultdict(list)
    for i, word in enumerate(transcript_words):
        index[word].append(i)
    return dict(index)


def _build_rarity_fn(transcript_words):
    freq = Counter(transcript_words)
    n = len(transcript_words)
    def rarity(word):
        return math.log(n / (1 + freq.get(word, 0)))
    return rarity


def choose_anchor(target_words, transcript_words, rarity_fn):
    vocab = set(transcript_words)
    candidates = [w for w in target_words if w in vocab]
    if not candidates:
        return None
    return max(candidates, key=rarity_fn)


def rare_anchor_fuzzy_search(transcript_words, target_words, score_fn=difflib_score,
                              extra_context=FUZZY_EXTRA_CONTEXT, length_tolerance=FUZZY_LENGTH_TOLERANCE,
                              inverted_index=None):
    if not target_words:
        return []

    rarity_fn = _build_rarity_fn(transcript_words)  # recomputed every call, no stale state
    anchor = choose_anchor(target_words, transcript_words, rarity_fn)
    if anchor is None:
        return []

    anchor_offset = target_words.index(anchor)
    n = len(target_words)

    if inverted_index is not None:
        anchor_positions = inverted_index.get(anchor, [])
    else:
        anchor_positions = [i for i, w in enumerate(transcript_words) if w == anchor]

    results = []
    for anchor_index in anchor_positions:
        expected_start = anchor_index - anchor_offset
        start = max(0, expected_start - extra_context)
        end = min(len(transcript_words), expected_start + n + extra_context)
        region = transcript_words[start:end]

        lo, hi = max(1, n - length_tolerance), n + length_tolerance
        best = None
        for wlen in range(lo, hi + 1):
            if wlen > len(region):
                continue
            for i in range(len(region) - wlen + 1):
                window = region[i:i + wlen]
                score = score_fn(target_words, window)
                if best is None or score > best["score"]:
                    best = {"start_index": start + i, "end_index": start + i + wlen,
                            "score": score, "method": "rare_anchor_fuzzy", "anchor": anchor}
        if best:
            results.append(best)

    results.sort(key=lambda r: r["score"], reverse=True)
    return results


In [ ]:
def search_dialogue(transcript_words, target_words, method="rare_anchor_fuzzy",
                     score_fn_name="difflib", inverted_index=None):
    score_fn = get_score_fn(score_fn_name) if method != "exact" else None
    if method == "exact":
        return exact_phrase_search(transcript_words, target_words)
    if method == "fuzzy":
        return fuzzy_sliding_window_search(transcript_words, target_words, score_fn=score_fn)
    if method == "rare_anchor_fuzzy":
        return rare_anchor_fuzzy_search(transcript_words, target_words, score_fn=score_fn,
                                         inverted_index=inverted_index)
    raise ValueError(f"Unknown method: {method}")


### 5.1 Regression tests (offline, real execution)

1. Stale-state check (carried over from v2): rarity is computed fresh
   per call.
2. Inverted index correctness: results with and without a prebuilt
   index must be identical.
3. RapidFuzz vs difflib: both should score an exact match as 1.0 and
   agree on which candidate is best for a clear case.

In [ ]:
_video_a = normalize_text(
    "my at my mind at my mind the weather today is fine the cat sat on the mat "
    "my mind rebels at stagnation the dog ran fast my mind is at home"
).split()
_video_b = normalize_text(
    "rebels rebels rebels rebels rebels rebels rebels rebels "
    "my my my mind mind mind at at at "
    "my mind rebels at stagnation"
).split()
_target = normalize_text("my mind rebels at stagnation").split()

_result_a = rare_anchor_fuzzy_search(_video_a, _target)
_result_b = rare_anchor_fuzzy_search(_video_b, _target)
assert _result_a[0]["anchor"] == "rebels"
assert _result_b[0]["anchor"] == "stagnation"
print("PASS: rarity is per-call, not stale (carried over from v2)")

_index_a = build_inverted_index(_video_a)
_result_a_indexed = rare_anchor_fuzzy_search(_video_a, _target, inverted_index=_index_a)
assert _result_a_indexed[0]["start_index"] == _result_a[0]["start_index"]
assert abs(_result_a_indexed[0]["score"] - _result_a[0]["score"]) < 1e-9
print("PASS: inverted-index lookup produces identical results to the linear scan")

_misspelled = normalize_text("my mind rebbels at stagnashun").split()
_diff_result = rare_anchor_fuzzy_search(_video_a, _misspelled, score_fn=difflib_score)
_rf_result = rare_anchor_fuzzy_search(_video_a, _misspelled, score_fn=rapidfuzz_score)
assert _diff_result[0]["start_index"] == _rf_result[0]["start_index"], "both scorers should agree on the best window here"
print(f"PASS: difflib and rapidfuzz agree on best match (difflib={_diff_result[0]['score']:.3f}, rapidfuzz={_rf_result[0]['score']:.3f})")


## 6. Timestamp → frame

In [ ]:
def timestamp_to_frame(timestamp: float, fps: float) -> int:
    return int(round(timestamp * fps))


def extract_frame(video_path: Path, timestamp: float, output_path: Path) -> Path:
    cmd = ["ffmpeg", "-y", "-i", str(video_path), "-ss", f"{timestamp:.6f}",
           "-frames:v", "1", "-q:v", "2", str(output_path)]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    return output_path


## 7. End-to-end function — audio-first, DB-backed, deferred video download

The full video (`download_video`) is only called **after** a match is
found — i.e. right before frame extraction, not up front. On a
fingerprint cache hit, ASR is skipped entirely.

In [ ]:
def find_dialogue(video_url: str, target_dialogue: str, method: str = "rare_anchor_fuzzy",
                   score_fn_name: str = "difflib", model_size: str = "small", top_k: int = 5,
                   use_inverted_index: bool = True) -> dict:
    conn = get_db()
    record = get_or_create_video_record(conn, video_url)
    video_id = record["video_id"]
    fps = record["fps"]

    if not record["has_audio"]:
        return {"success": False, "video_url": video_url, "message": "Video has no audio track."}

    audio_path = AUDIO_DIR / f"{video_id}.wav"
    if not audio_path.exists():
        audio_path = download_audio_only(video_url)

    transcript = transcribe_video(conn, video_id, audio_path, model_size=model_size)
    if not transcript:
        return {"success": False, "video_url": video_url, "message": "No speech detected."}

    transcript_words = build_word_index(transcript)
    target_words = normalize_text(target_dialogue).split()
    if not target_words:
        raise ValueError("Target dialogue is empty.")

    inverted_index = build_inverted_index(transcript_words) if use_inverted_index else None
    raw_results = search_dialogue(transcript_words, target_words, method=method,
                                   score_fn_name=score_fn_name, inverted_index=inverted_index)
    if not raw_results:
        return {"success": False, "video_url": video_url, "query": target_dialogue,
                "method": method, "message": "Dialogue was not found."}

    # Only now — once we know there's a match worth a frame — download the full video.
    video_path = download_video(video_url)

    matches = []
    for rank, result in enumerate(raw_results[:top_k], start=1):
        start_index, end_index = result["start_index"], result["end_index"]
        start_time = float(transcript[start_index]["start"])
        end_time = float(transcript[end_index - 1]["end"])
        start_frame = timestamp_to_frame(start_time, fps)
        end_frame = timestamp_to_frame(end_time, fps)

        frame_path = FRAME_DIR / f"{video_id}_frame_{start_frame}.jpg"
        if not frame_path.exists():
            extract_frame(video_path, start_time, frame_path)

        matched_text = " ".join(transcript[i]["word"] for i in range(start_index, end_index))
        matches.append({
            "rank": rank, "matched_text": matched_text,
            "start_timestamp": start_time, "end_timestamp": end_time,
            "start_frame": start_frame, "end_frame": end_frame,
            "score": float(result["score"]), "frame_path": str(frame_path),
            "anchor": result.get("anchor"),
        })

    final_result = {
        "success": True,
        "video": {"url": video_url, "duration_seconds": record["duration"], "fps": fps},
        "query": {"dialogue": target_dialogue, "normalized": " ".join(target_words),
                   "method": method, "score_fn": score_fn_name, "model_size": model_size},
        "matches": matches,
    }
    result_path = RESULT_DIR / f"{video_id}_{model_size}_result.json"
    result_path.write_text(json.dumps(final_result, indent=2))
    final_result["result_file"] = str(result_path)
    return final_result


⚠️ Not executed here — network restriction, as in v2.

In [ ]:
result = find_dialogue(DEFAULT_VIDEO_URL, DEFAULT_TARGET_DIALOGUE)
print(json.dumps(result, indent=2))


In [ ]:
if result.get("success"):
    best = result["matches"][0]
    print(f"Timestamp : {best['start_timestamp']:.3f}s")
    print(f"Frame     : {best['start_frame']}")
    print(f"Text      : \"{best['matched_text']}\"")
    display(Image.open(best["frame_path"]))


## 8. Accuracy benchmark harness (generalized)

Same idea as v2's `run_accuracy_benchmark`, generalized to accept named
variants (method + scorer + index choice) so §9 and §10 can both drive
it with different variant lists.

In [ ]:
def run_variant_benchmark(transcript, known_dialogues, variants, runs=3):
    '''variants: list of dicts, each {label, method, score_fn_name, use_index}'''
    transcript_words = build_word_index(transcript)
    inverted_index = build_inverted_index(transcript_words)
    rows = []

    for variant in variants:
        label = variant["label"]
        method = variant["method"]
        score_fn_name = variant.get("score_fn_name", "difflib")
        idx = inverted_index if variant.get("use_index") else None

        correct, errors, times = 0, [], []
        for dialogue, expected_start, tolerance in known_dialogues:
            target_words = normalize_text(dialogue).split()

            t0 = time.perf_counter()
            for _ in range(runs):
                results = search_dialogue(transcript_words, target_words, method=method,
                                           score_fn_name=score_fn_name, inverted_index=idx)
            elapsed_ms = (time.perf_counter() - t0) / runs * 1000
            times.append(elapsed_ms)

            if results:
                predicted_start = float(transcript[results[0]["start_index"]]["start"])
                error = abs(predicted_start - expected_start)
                errors.append(error)
                if error <= tolerance:
                    correct += 1
            else:
                errors.append(float("nan"))

        rows.append({
            "variant": label,
            "top1_accuracy": correct / len(known_dialogues) if known_dialogues else float("nan"),
            "mean_abs_error_s": pd.Series(errors).mean(),
            "mean_time_ms": pd.Series(times).mean(),
        })

    return pd.DataFrame(rows).sort_values("mean_time_ms").reset_index(drop=True)


## 9. Model-size comparison — fixed

v2's version silently reused a cached transcript for `small`, making it
look near-instant. `force=True` by default here so every size is timed
honestly; pass `force=False` deliberately if you want to test cache
behavior itself.

In [ ]:
def compare_model_sizes(conn, video_id, audio_path, model_sizes=("tiny", "base", "small", "medium"),
                         known_dialogues=None, method="rare_anchor_fuzzy", force=True):
    known_dialogues = known_dialogues or []
    rows = []
    for model_size in model_sizes:
        t0 = time.perf_counter()
        transcript = transcribe_video(conn, video_id, audio_path, model_size=model_size, force=force)
        transcribe_s = time.perf_counter() - t0

        acc_df = run_variant_benchmark(
            transcript, known_dialogues,
            variants=[{"label": method, "method": method, "score_fn_name": "difflib", "use_index": True}],
        )
        row = acc_df.iloc[0].to_dict()
        row.update({"model_size": model_size, "transcribe_time_s": transcribe_s, "word_count": len(transcript)})
        rows.append(row)

    return pd.DataFrame(rows)[["model_size", "word_count", "transcribe_time_s",
                                "top1_accuracy", "mean_abs_error_s", "mean_time_ms"]]


# KNOWN_DIALOGUES = [("My mind rebels at stagnation", 325.0, 2.0)]
# conn = get_db()
# model_comparison_df = compare_model_sizes(conn, video_id, audio_path, known_dialogues=KNOWN_DIALOGUES)
# model_comparison_df


## 10. Comparative search-method benchmark

Every method × scorer × index combination, run against the same
transcript and `KNOWN_DIALOGUES`. This is the analysis requested:
does RapidFuzz actually beat difflib on speed, does the inverted index
matter at this transcript size, and where does the embedding scorer
sit on the speed/robustness tradeoff.

In [ ]:
SEARCH_VARIANTS = [
    {"label": "exact", "method": "exact"},
    {"label": "fuzzy + difflib", "method": "fuzzy", "score_fn_name": "difflib"},
    {"label": "fuzzy + rapidfuzz", "method": "fuzzy", "score_fn_name": "rapidfuzz"},
    {"label": "rare_anchor + difflib (linear scan)", "method": "rare_anchor_fuzzy", "score_fn_name": "difflib", "use_index": False},
    {"label": "rare_anchor + difflib (inverted index)", "method": "rare_anchor_fuzzy", "score_fn_name": "difflib", "use_index": True},
    {"label": "rare_anchor + rapidfuzz (inverted index)", "method": "rare_anchor_fuzzy", "score_fn_name": "rapidfuzz", "use_index": True},
]
if EMBEDDING_AVAILABLE:
    SEARCH_VARIANTS.append(
        {"label": "rare_anchor + embedding (inverted index)", "method": "rare_anchor_fuzzy",
         "score_fn_name": "embedding", "use_index": True}
    )

# KNOWN_DIALOGUES = [("My mind rebels at stagnation", 325.0, 2.0)]
# search_comparison_df = run_variant_benchmark(transcript, KNOWN_DIALOGUES, SEARCH_VARIANTS)
# search_comparison_df


### 10.1 Sanity check on synthetic data (offline, real execution)

Confirms the comparative harness itself works end-to-end before you
point it at a real transcript — including that the inverted-index and
linear-scan variants of `rare_anchor_fuzzy` agree, and that every
variant finds the known phrase.

In [ ]:
_synthetic_transcript = [
    {"word": w, "start": float(i), "end": float(i) + 0.9}
    for i, w in enumerate(normalize_text(
        "the weather today is fine the cat sat on the mat "
        "my mind rebels at stagnation the dog ran fast"
    ).split())
]
_synthetic_known = [("My mind rebels at stagnation", 12.0, 1.5)]

_comparison = run_variant_benchmark(_synthetic_transcript, _synthetic_known, SEARCH_VARIANTS)
print(_comparison.to_string(index=False))

assert (_comparison["top1_accuracy"] == 1.0).all(), "every variant should find the known phrase on this clean synthetic transcript"
print("\nPASS: all search variants agree and find the correct match on synthetic data")
